# MUSANGA V3.1 — NB06B: Spatial context in 64-dimensional Earth embeddings

**Companion notebook to the validated clean NB06 Block 1 (Sentinel-2 baseline and figures 1.D).** This notebook does **not** overwrite that baseline or its figures. Its purpose is a controlled study of **spatial embedding context**, using the same UAV-derived fractional-cover pseudo-label target. The original NB06 source was not accessible when this revision was prepared, so this is a separate self-contained runnable section, **not a verified verbatim replacement of NB06**.

### Scientific design

1. Embedding **point** baseline (64 native embedding features).
2. Embedding **local means and heterogeneity**, plus directional spatial contrast, at *30, 60, and 120 m* context radii.
3. The **same model and folds** for all treatments. Context features are calculated **only from embedding rasters, never targets**.
4. **Spatially blocked within-site validation** and **strict leave-one-mosaic-out** validation, with MOS_1_2 and MOS_5_6 separate.
5. **Equal-mosaic summaries**, pooled and per-site accuracy, high-fraction bias, paired error differences, diagnostics and feature auditing.

**Caution:** embedding pixel resolution and support must be checked. A smooth descriptor is not automatically a native high-resolution signal. The experiment tests contextual representation, not tree crown recognition or independent validation of field cover.

**Run order:** configuration → preflight → context cache → design table → validation → figures → interpretation. Initial execution requires local/Drive raster files and setting `SITE_INPUTS` below.


## 0. Runtime and explicit inputs

In [ ]:
# Colab-safe; install only missing packages.
import importlib, subprocess, sys
for mod,pkg in [('rasterio','rasterio'),('sklearn','scikit-learn'),('scipy','scipy'),('pandas','pandas'),('matplotlib','matplotlib'),('joblib','joblib'),('tqdm','tqdm'),('pyarrow','pyarrow')]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
from pathlib import Path
import os, json, hashlib, warnings, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from scipy.ndimage import uniform_filter
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
from joblib import dump

# Uncomment if in Colab:
# from google.colab import drive; drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')  # adjust if necessary
OUT = ROOT/'CLEAN_NB06'/'02_embedding_spatial'
OUT.mkdir(parents=True, exist_ok=True)

# Fill in each site with its EXISTING target and aligned 64-band embedding GeoTIFF.
# No download or re-extraction is performed in this notebook.
SITE_INPUTS = {
    'Baego':    {'target':None, 'embedding':None, 'roi':None},
    'MOS_1_2':  {'target':None, 'embedding':None, 'roi':None},
    'MOS_5_6':  {'target':None, 'embedding':None, 'roi':None},
    'Yangambi': {'target':None, 'embedding':None, 'roi':None},
    'Yoko':     {'target':None, 'embedding':None, 'roi':None},
}
# target is single-band fraction [0,1], or percentage [0,100] (detected and logged).
# roi, if supplied, is a one-band 0/1 mask aligned with target/embedding.
RADII_M = [30,60,120]
TARGET_GRID_STEP_M = None # Auto from geotransform; must be projected in metres.
MAX_POINTS_PER_SITE = 15000
SPATIAL_BLOCK_M = 500
SPATIAL_GAP_M = 120 # conservative exclusion around holdout BLOCK boundary (applied below)
N_SPLITS = 4
SEED = 47
HIGH_FRACTION = 0.40
MIN_CONTEXT_SUPPORT = 0.70
print('Output directory:', OUT)


## 1. Preflight — strict raster/grid/target contract

In [ ]:
def check_inputs(inputs):
    rows=[]
    for site,paths in inputs.items():
        for key in ('target','embedding'):
            if not paths.get(key): raise ValueError(f'{site}: configure {key} file path in SITE_INPUTS before running.')
            if not Path(paths[key]).exists(): raise FileNotFoundError(f'{site}: missing {key}: {paths[key]}')
        with rasterio.open(paths['target']) as t, rasterio.open(paths['embedding']) as e:
            if t.count != 1 or e.count != 64: raise ValueError(f'{site}: expected target 1 band and embedding 64 bands; got {t.count}, {e.count}')
            if not t.crs or not t.crs.is_projected: raise ValueError(f'{site}: geographic coordinate grid; reproject explicitly to a metre-based CRS first')
            if t.crs != e.crs or t.transform != e.transform or t.shape != e.shape:
                raise ValueError(f'{site}: target/embedding grids not identical. Explicitly align beforehand; never silently resample.')
            if t.crs.linear_units.lower() not in ['metre','meter','metres','meters']:
                raise ValueError(f'{site}: projected CRS linear units not metres: {t.crs.linear_units}')
            if paths.get('roi'):
                with rasterio.open(paths['roi']) as r:
                    if r.count != 1 or r.crs!=t.crs or r.transform!=t.transform or r.shape!=t.shape:
                        raise ValueError(f'{site}: ROI grid is not identical to target grid')
            rows.append(dict(site=site, width=t.width,height=t.height,cell_x=abs(t.transform.a),cell_y=abs(t.transform.e),crs=str(t.crs),pixel_area_m2=abs(t.transform.a*t.transform.e),embedding_dtype=e.dtypes[0]))
    return pd.DataFrame(rows)
PREFLIGHT = check_inputs(SITE_INPUTS)
display(PREFLIGHT)
PREFLIGHT.to_csv(OUT/'preflight.csv',index=False)


## 2. Extract native embedding + multi-scale spatial context

Context is computed on the original, **aligned embedding grid**, before point sampling. For each of 64 bands and each radius, derive local mean and SD plus N−S and E−W local mean contrasts. A rolling square window approximates the specified distance; we record its actual radius in pixels. Masked pixels are excluded. The contrast is signed in the *raster coordinate orientation*, so it is sensitive to rotation; all scenes should be north-up. Target labels are not used in any context operation.

**Memory control:** scenes are processed one embedding band at a time; only sampled features are retained. For large mosaics, run against ROI-clipped rasters rather than the full forest mosaic. This avoids caching entire multi-band context cubes.


In [ ]:
def mean_valid(a, valid, window):
    count=uniform_filter(valid.astype('float32'),size=window,mode='constant',cval=0)
    mass=uniform_filter(np.where(valid,a,0).astype('float32'),size=window,mode='constant',cval=0)
    return np.divide(mass,count,out=np.full_like(mass,np.nan),where=count>1e-6),count

def context_one_band(a,valid,radpx):
    # Uniform square window, radius radpx. SD includes all unmasked cells.
    win=2*radpx+1
    mu,support=mean_valid(a,valid,win)
    second,_=mean_valid(np.where(valid,a*a,0),valid,win)
    sd=np.sqrt(np.maximum(0, second-mu*mu))
    # Two opposing half-window means, excluding central stripe.
    def shift_half_mean(axis,negative):
        # Use a directional offset of +/- half radius; keeps kernel compact and inexpensive.
        from scipy.ndimage import shift
        delta = float(radpx)/2
        shifts=[0.,0.]
        shifts[axis] = (-delta if negative else delta)
        # Shift numerator and denominator independently, avoid nodata pollution.
        z=uniform_filter(np.where(valid,a,0).astype('float32'),size=win,mode='constant',cval=0)
        c=uniform_filter(valid.astype('float32'),size=win,mode='constant',cval=0)
        zz=shift(z, shifts, order=1,mode='constant',cval=0)
        cc=shift(c, shifts, order=1,mode='constant',cval=0)
        return np.divide(zz,cc,out=np.full_like(zz,np.nan),where=cc>1e-6)
    # North/south and west/east comparisons of offset box means.
    ns=shift_half_mean(0,True)-shift_half_mean(0,False)
    ew=shift_half_mean(1,True)-shift_half_mean(1,False)
    trusted=support >= MIN_CONTEXT_SUPPORT
    for z in [mu,sd,ns,ew]: z[~trusted]=np.nan
    return mu,sd,ns,ew,support

def process_site(site,paths):
    rng=np.random.default_rng(SEED)
    with rasterio.open(paths['target']) as t, rasterio.open(paths['embedding']) as e:
        y=t.read(1).astype('float32'); valid_target=np.isfinite(y)&(t.read_masks(1)>0)
        if paths.get('roi'):
            with rasterio.open(paths['roi']) as r: valid_target &= (r.read(1)>0)&(r.read_masks(1)>0)
        yy=y[valid_target]
        if yy.size==0: raise ValueError(f'{site}: no valid target pixels')
        ymax=float(np.nanpercentile(yy,99.99)); ymin=float(np.nanmin(yy))
        if ymin < -0.00001 or ymax>100.01: raise ValueError(f'{site}: invalid target range [{ymin}, {ymax}]')
        units='percent' if ymax>1.001 else 'fraction'
        if units=='percent': y=y/100.
        # Only use labelled locations where the complete 64-band center is valid.
        centers=valid_target.copy()
        for k in range(1,65):
            arr=e.read(k,masked=True)
            centers &= ~np.ma.getmaskarray(arr) & np.isfinite(np.asarray(arr.data))
        rr,cc=np.nonzero(centers)
        if len(rr)>MAX_POINTS_PER_SITE:
            take=rng.choice(len(rr),MAX_POINTS_PER_SITE,replace=False)
            rr,cc=rr[take],cc[take]
        if not len(rr): raise ValueError(f'{site}: no co-valid target/embedding cells')
        xs,ys=rasterio.transform.xy(t.transform,rr,cc,offset='center')
        tab=pd.DataFrame({'site':site,'row':rr,'col':cc,'x':np.asarray(xs),'ycoord':np.asarray(ys),'target':y[rr,cc]})
        tab['block_x']=np.floor(tab.x/SPATIAL_BLOCK_M).astype('int32')
        tab['block_y']=np.floor(tab.ycoord/SPATIAL_BLOCK_M).astype('int32')
        tab['group']=tab.site+'__'+tab.block_x.astype(str)+'__'+tab.block_y.astype(str)
        px=abs(t.transform.a); py=abs(t.transform.e)
        if abs(px-py)/max(px,py)>.05: raise ValueError(f'{site}: non-square pixels; revise context windows')
        if abs(t.transform.b)>1e-9 or abs(t.transform.d)>1e-9: raise ValueError(f'{site}: rotated transform unsupported')
        for k in tqdm(range(1,65),desc=f'{site}: embedding bands'):
            aa=e.read(k,masked=True)
            a=np.asarray(aa.data,dtype='float32')
            valid=~np.ma.getmaskarray(aa) & np.isfinite(a)
            tab[f'E{k:02d}']=a[rr,cc]
            for radius in RADII_M:
                rp=max(1,round(radius/px)); mean,sd,ns,ew,support=context_one_band(a,valid,rp)
                prefix=f'R{radius}_E{k:02d}'
                tab[f'{prefix}_mean']=mean[rr,cc]
                tab[f'{prefix}_sd']=sd[rr,cc]
                tab[f'{prefix}_ns']=ns[rr,cc]
                tab[f'{prefix}_ew']=ew[rr,cc]
        path=OUT/f'{site}_embedding_context.parquet'
        tab.to_parquet(path,index=False)
        print(site, len(tab),'sampled pixels; target units:', units, 'saved:',path)
        return dict(site=site,n=len(tab),target_input_units=units,pixel_m=px,output=str(path))

manifest=[]
for site,inputs in SITE_INPUTS.items():
    cache=OUT/f'{site}_embedding_context.parquet'
    if cache.exists():
        print('CACHE EXISTS:',cache,'— reused (delete manually if source rasters/configuration changed)')
        manifest.append(dict(site=site,output=str(cache),cached=True))
    else: manifest.append(process_site(site,inputs))
with open(OUT/'context_manifest.json','w') as f:json.dump({'sites':manifest,'radii_m':RADII_M,'seed':SEED,'max_points_per_site':MAX_POINTS_PER_SITE,'block_m':SPATIAL_BLOCK_M},f,indent=2)


## 3. Assemble features and audit coverage

In [ ]:
df=pd.concat([pd.read_parquet(OUT/f'{s}_embedding_context.parquet') for s in SITE_INPUTS],ignore_index=True)
POINT=[f'E{k:02d}' for k in range(1,65)]
CONTEXT={r:[f'R{r}_E{k:02d}_{stat}' for k in range(1,65) for stat in ('mean','sd','ns','ew')] for r in RADII_M}
FEATURES={'EMB64_point': POINT}
for r in RADII_M: FEATURES[f'EMB64_point_plus_context_{r}m']=POINT+CONTEXT[r]
FEATURES['EMB64_point_plus_context_all']=POINT+sum((CONTEXT[r] for r in RADII_M),[])
print('Rows by mosaic');display(df.groupby('site').agg(n=('target','size'),mean_target=('target','mean'),high_n=('target',lambda x:(x>=HIGH_FRACTION).sum())))
print('Finite feature fraction by family:')
display(pd.DataFrame([dict(model=m,n_features=len(cols),finite_frac=float(np.isfinite(df[cols].to_numpy(dtype='float32')).mean())) for m,cols in FEATURES.items()]))
if df.target.min()<0 or df.target.max()>1:raise ValueError('Target not fractional')


## 4. Fixed validation — spatial blocks + gap and strict leave-one-mosaic-out

Spatial folds are based on metric blocks. To reduce neighbouring training/test contamination, **training points within `SPATIAL_GAP_M` of any test point are excluded**. Validation is split **by mosaic**; no random-pixel CV. For LOSO, entire individual mosaics are held out. The per-site fold assessment is the primary comparison, while pooled metrics are descriptive.


In [ ]:
from scipy.spatial import cKDTree

def make_folds(data, mode='blocked'):
    folds=[]
    if mode=='loso':
        for site in sorted(data.site.unique()):
            te=np.flatnonzero(data.site.to_numpy()==site);tr=np.flatnonzero(data.site.to_numpy()!=site)
            folds.append((f'LOSO_{site}',tr,te))
    else:
        for site in sorted(data.site.unique()):
            loc=np.flatnonzero(data.site.to_numpy()==site)
            groups=data.iloc[loc].group.to_numpy()
            if len(np.unique(groups)) < N_SPLITS: raise ValueError(f'{site}: only {len(np.unique(groups))} blocks; fewer than {N_SPLITS}')
            gkf=GroupKFold(n_splits=N_SPLITS)
            for j,(a,b) in enumerate(gkf.split(loc,groups=groups)):
                te=loc[b]; tr=loc[a]
                # Same-site buffer. cKDTree is metric, efficiently removes nearby training points.
                tree=cKDTree(data.iloc[te][['x','ycoord']].to_numpy())
                nearest=tree.query(data.iloc[tr][['x','ycoord']].to_numpy(),k=1,workers=-1)[0]
                tr=tr[nearest>=SPATIAL_GAP_M]
                if len(tr)<100 or len(te)<20:raise ValueError(f'{site}: insufficient buffered train/test samples')
                folds.append((f'{site}_block{j}',tr,te))
    return folds
FOLDS={kind:make_folds(df,kind) for kind in ('blocked','loso')}
print({k:[(name,len(tr),len(te)) for name,tr,te in v] for k,v in FOLDS.items()})


## 5. Model comparison — identical estimator, paired predictions

In [ ]:
def fit_model(x,y):
    return Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=False)),
        ('model',HistGradientBoostingRegressor(max_iter=220,learning_rate=.05,max_leaf_nodes=15,l2_regularization=5,random_state=SEED))]).fit(x,y)

def scores(d):
    y=d.target.to_numpy();p=d.pred.to_numpy();n=len(y)
    return dict(n=n,R2=r2_score(y,p) if n>1 and np.var(y)>0 else np.nan,
      MAE_pp=100*mean_absolute_error(y,p),RMSE_pp=100*np.sqrt(mean_squared_error(y,p)),
      bias_pp=100*np.mean(p-y),high_n=int((y>=HIGH_FRACTION).sum()),
      high_bias_pp=100*np.mean(p[y>=HIGH_FRACTION]-y[y>=HIGH_FRACTION]) if np.any(y>=HIGH_FRACTION) else np.nan)

preds=[]
for mode,folds in FOLDS.items():
    for model_name,cols in FEATURES.items():
        for fold,tr,te in tqdm(folds,desc=f'{mode}:{model_name}'):
            xtr=df.iloc[tr][cols].replace([np.inf,-np.inf],np.nan)
            xte=df.iloc[te][cols].replace([np.inf,-np.inf],np.nan)
            # Columns with zero usable train data are excluded consistently inside this fold.
            usable=xtr.notna().any(axis=0)
            use=list(usable.index[usable])
            if len(use)<20: raise ValueError(f'{fold}: insufficient valid features')
            est=fit_model(xtr[use],df.iloc[tr].target)
            p=np.clip(est.predict(xte[use]),0,1)
            o=df.iloc[te][['site','row','col','x','ycoord','target','group']].copy()
            o['pred']=p;o['model']=model_name;o['validation']=mode;o['fold']=fold
            preds.append(o)
            print(f'{mode:8s} {fold:24s} {model_name:36s} MAE={100*np.mean(np.abs(p-o.target)):.2f} pp',flush=True)
PRED=pd.concat(preds,ignore_index=True)
PRED.to_parquet(OUT/'spatial_embedding_predictions.parquet',index=False)
res=pd.DataFrame([dict(validation=v,model=m,site=s,**scores(g)) for (v,m,s),g in PRED.groupby(['validation','model','site'])])
res.to_csv(OUT/'metrics_by_site.csv',index=False)
display(res.round(3))


## 6. Equal-mosaic assessment and improvement against EMB64 point

In [ ]:
metrics=['MAE_pp','RMSE_pp','bias_pp','high_bias_pp']
summary=res.groupby(['validation','model']).agg(**{'mean_'+k:(k,'mean') for k in metrics},mean_site_R2=('R2','mean'),n_sites=('site','nunique')).reset_index()
summary.to_csv(OUT/'equal_mosaic_summary.csv',index=False)
display(summary.round(3))
base=summary[summary.model=='EMB64_point'].set_index('validation')
changes=[]
for _,r in summary.iterrows():
    q=base.loc[r.validation]
    changes.append(dict(validation=r.validation,model=r.model,delta_MAE_pp=r.mean_MAE_pp-q.mean_MAE_pp,delta_high_bias_pp=r.mean_high_bias_pp-q.mean_high_bias_pp))
change=pd.DataFrame(changes);display(change.round(3));change.to_csv(OUT/'paired_model_deltas.csv',index=False)


## 7. Diagnostic figures (held-out only)

In [ ]:
for mode in ('blocked','loso'):
    sub=res[res.validation==mode]
    fig,ax=plt.subplots(figsize=(11,4))
    sites=sorted(sub.site.unique()); models=list(FEATURES);xx=np.arange(len(sites));w=.8/len(models)
    for i,m in enumerate(models):
        q=sub[sub.model==m].set_index('site').reindex(sites)
        ax.bar(xx-.4+(i+.5)*w,q.MAE_pp,width=w,label=m)
    ax.set_xticks(xx,sites);ax.set_ylabel('Held-out MAE (percentage points)');ax.set_title(f'{mode.upper()} — errors by held-out mosaic');ax.legend(fontsize=7,ncol=2);fig.tight_layout();fig.savefig(OUT/f'{mode}_MAE_by_mosaic.png',dpi=180);plt.show()

# Paired out-of-fold predictions, displayed equally across mosaics using per-site transparency.
for mode in ('blocked','loso'):
    sub=PRED[PRED.validation==mode]
    models=list(FEATURES)
    fig,axes=plt.subplots(1,len(models),figsize=(4.0*len(models),4),sharex=True,sharey=True)
    for ax,m in zip(axes,models):
        q=sub[sub.model==m]
        for site,g in q.groupby('site'):
            ax.scatter(100*g.target,100*g.pred,s=2,alpha=.12,label=site,rasterized=True)
        ax.plot([0,100],[0,100],ls='--',lw=1,color='black')
        ax.set_title(m.replace('EMB64_point_plus_',''),fontsize=9);ax.set_xlim(0,100);ax.set_ylim(0,100)
        ax.set_xlabel('UAV pseudo-label (%)')
    axes[0].set_ylabel('Held-out prediction (%)');axes[-1].legend(markerscale=5,fontsize=7)
    fig.suptitle(f'{mode.upper()} spatial validation (unbalanced scatter; use equal-mosaic stats for comparisons)')
    fig.tight_layout();fig.savefig(OUT/f'{mode}_scatter.png',dpi=180);plt.show()


## 8. Interpretation and decision rule

Inspect **both** validation regimes. A contextual model is worthwhile only if it improves equal-mosaic *blocked* MAE and does not markedly worsen strict LOSO MAE or high-abundance bias. Improvements that appear only in MOS-heavy pooled plots are not sufficient.

Watch for saturation above 40% in MOS and for artificially smooth spatial predictions. A very high within-site performance paired with poor LOSO indicates local spatial interpolation rather than transferable abundance retrieval. Predictions are referenced to UAV pseudo-labels; uncertainty in UAV segmentation is not represented here.

**Important scientific limitation:** the 64 embedding bands may already encode considerable spatial context. Neighbour averaging/directional contrast does not prove the source model is sensitive to Musanga crown structure, and its gain should be interpreted as an empirical incremental gain only.


In [ ]:
q=change[change.model!='EMB64_point'].pivot(index='model',columns='validation',values='delta_MAE_pp')
display(q.round(3))
print('Negative delta MAE = improved over point-only embedding baseline.')
print('Files available:',*[p.name for p in sorted(OUT.glob('*'))],sep='\n - ')
